# Pattern Mining

## Präambel

Am Anfang eines Skripts werden meist alle Softwarepakete geladen, die darin verwendet werden. In diesem Skript machen wir eine Ausnahme, um uns schrittweise an die Logik von Python zu gewöhnen.

Zunächst laden wir das Paket `pandas`, das wir für die Bearbeitung von Datensätzen verwenden.

In [1]:
# Lade das Paket pandas und vergib den Alias pd dafür (das erspart den Tippaufwand - echt!)
import pandas as pd

## Daten

Wir verwenden einen Datensatz von Kaggle: https://www.kaggle.com/datasets/lissetteg/ecommerce-dataset

Kaggle (https://www.kaggle.com/) ist eine Plattform, über die Datensätze für die Entwicklung von Modellen zur Verfügung gestellt werden. Die Webseite ist vor allem als Host von Wettbewerben mit bemerkenswert hohen Preisgeldern bekannt. Die Modelle werden auch geteilt.

Um den Datensatz direkt in Python zu laden, verwenden wir das Paket `kagglehub`:

In [2]:
# Lade das Paket kagglehub, aber diesmal ohne Alias
import kagglehub

ModuleNotFoundError: No module named 'kagglehub'

In [3]:
# Lade die aktuelle Version des Datensatzes herunter
path = kagglehub.dataset_download("lissetteg/ecommerce-dataset") # Pfad zum heruntergeladenen Datensatz

# Verwende die Funktion read_csv aus dem pandas-Paket
df = pd.read_csv(path + '/data-2.csv')

In [7]:
# Wirf einen Blick auf den eingelesenen Datensatz
df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,12/9/2011 12:50,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,12/9/2011 12:50,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,12/9/2011 12:50,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,12/9/2011 12:50,4.15,12680.0,France


In [5]:
# Verteilungsmaße für die numerischen Spalten im Datensatz
df.describe()

,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


## Datenaufbereitung

Lösche alle Einträge, die nicht im Datensatz sein sollten. Das betrifft Rechnungsnummern, die ein "A" oder "C" enthalten.

In [6]:
df = df[~df['InvoiceNo'].str.contains('A|C')]
len(df) # Ergibt jetzt 532618

532618

Betrachte nur Daten aus Dänemark, Deutschland und Italien

In [9]:
# Wähle Daten aus drei Ländern
subset_df = df[df['Country'].isin(['Denmark', 'Germany', 'Italy'])]
subset_df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
1109,536527,22809,SET OF 6 T-LIGHTS SANTA,6,12/1/2010 13:04,2.95,12662.0,Germany
1110,536527,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,6,12/1/2010 13:04,2.55,12662.0,Germany
1111,536527,84945,MULTI COLOUR SILVER T-LIGHT HOLDER,12,12/1/2010 13:04,0.85,12662.0,Germany
1112,536527,22242,5 HOOK HANGER MAGIC TOADSTOOL,12,12/1/2010 13:04,1.65,12662.0,Germany
1113,536527,22244,3 HOOK HANGER MAGIC GARDEN,12,12/1/2010 13:04,1.95,12662.0,Germany
...,...,...,...,...,...,...,...,...
541801,581578,22993,SET OF 4 PANTRY JELLY MOULDS,12,12/9/2011 12:16,1.25,12713.0,Germany
541802,581578,22907,PACK OF 20 NAPKINS PANTRY DESIGN,12,12/9/2011 12:16,0.85,12713.0,Germany
541803,581578,22908,PACK OF 20 NAPKINS RED APPLES,12,12/9/2011 12:16,0.85,12713.0,Germany
541804,581578,23215,JINGLE BELL HEART ANTIQUE SILVER,12,12/9/2011 12:16,2.08,12713.0,Germany


Der Apriori-Algorithmus benötigt die Daten in einem speziellen Format, in das die Daten gebracht werden müssen.

Zähle pro Rechnung und Produktnamen die Mengen der gekauften Produkte:

In [10]:
subset_df = subset_df.groupby(['InvoiceNo', 'Description'])['Quantity'].sum()
subset_df

InvoiceNo  Description                        
536527     3 HOOK HANGER MAGIC GARDEN             12
           5 HOOK HANGER MAGIC TOADSTOOL          12
           5 HOOK HANGER RED MAGIC TOADSTOOL      12
           ASSORTED COLOUR LIZARD SUCTION HOOK    24
           CHILDREN'S CIRCUS PARADE MUG           12
                                                  ..
581578     SPOTTY BUNTING                          9
           VINTAGE DONKEY TAIL GAME                6
           WRAP ALPHABET POSTER                   25
           WRAP CIRCUS PARADE                     25
           WRAP RED APPLES                        25
Name: Quantity, Length: 10151, dtype: int64

Bringe die Daten in ein "breites" Format, wo für jedes Produkt eine eigene Spalte mit der pro Rechnung bestellten Menge enthalten ist.

In [11]:
# Eine Zeile pro Rechnung
# Wenn ein Produkt in einer Rechnung enthalten war, steht dort ein Wert, ansonsten NaN
subset_df = subset_df.unstack()
subset_df

Description,50'S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,I LOVE LONDON MINI BACKPACK,RED SPOT GIFT BAG LARGE,SET 2 TEA TOWELS I LOVE LONDON,SPACEBOY BABY GIFT SET,10 COLOUR SPACEBOY PEN,12 COLOURED PARTY BALLOONS,12 EGG HOUSE PAINTED WOOD,12 IVORY ROSE PEG PLACE SETTINGS,...,YULETIDE IMAGES GIFT WRAP SET,ZINC HEART T-LIGHT HOLDER,ZINC STAR T-LIGHT HOLDER,ZINC BOX SIGN HOME,ZINC FOLKART SLEIGH BELLS,ZINC HEART LATTICE T-LIGHT HOLDER,ZINC METAL HEART DECORATION,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS SMALL,ZINC WILLIE WINKIE CANDLE STICK
InvoiceNo,,,,,,,,,,,,,,,,,,,,,
536527,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
536840,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
536861,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
536967,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
536983,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
581266,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
581494,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
581570,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Generiere einen "sauberen" Index. Wird aus technischen Gründen benötigt.

In [12]:
subset_df = subset_df.reset_index()
subset_df

Description,InvoiceNo,50'S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,I LOVE LONDON MINI BACKPACK,RED SPOT GIFT BAG LARGE,SET 2 TEA TOWELS I LOVE LONDON,SPACEBOY BABY GIFT SET,10 COLOUR SPACEBOY PEN,12 COLOURED PARTY BALLOONS,12 EGG HOUSE PAINTED WOOD,...,YULETIDE IMAGES GIFT WRAP SET,ZINC HEART T-LIGHT HOLDER,ZINC STAR T-LIGHT HOLDER,ZINC BOX SIGN HOME,ZINC FOLKART SLEIGH BELLS,ZINC HEART LATTICE T-LIGHT HOLDER,ZINC METAL HEART DECORATION,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS SMALL,ZINC WILLIE WINKIE CANDLE STICK
0,536527,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,536840,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,536861,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,536967,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,536983,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
508,581266,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
509,581494,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
510,581570,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
511,581574,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Ersetze die NaN Werte mit 0, damit die Funktion damit arbeiten kann

In [13]:
subset_df = subset_df.fillna(0)
subset_df

Description,InvoiceNo,50'S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,I LOVE LONDON MINI BACKPACK,RED SPOT GIFT BAG LARGE,SET 2 TEA TOWELS I LOVE LONDON,SPACEBOY BABY GIFT SET,10 COLOUR SPACEBOY PEN,12 COLOURED PARTY BALLOONS,12 EGG HOUSE PAINTED WOOD,...,YULETIDE IMAGES GIFT WRAP SET,ZINC HEART T-LIGHT HOLDER,ZINC STAR T-LIGHT HOLDER,ZINC BOX SIGN HOME,ZINC FOLKART SLEIGH BELLS,ZINC HEART LATTICE T-LIGHT HOLDER,ZINC METAL HEART DECORATION,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS SMALL,ZINC WILLIE WINKIE CANDLE STICK
0,536527,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,536840,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,536861,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,536967,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,536983,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
508,581266,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
509,581494,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
510,581570,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
511,581574,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Definiere die Spalte 'InvoiceNo' als Index

In [14]:
subset_df = subset_df.set_index('InvoiceNo')
subset_df

Description,50'S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,I LOVE LONDON MINI BACKPACK,RED SPOT GIFT BAG LARGE,SET 2 TEA TOWELS I LOVE LONDON,SPACEBOY BABY GIFT SET,10 COLOUR SPACEBOY PEN,12 COLOURED PARTY BALLOONS,12 EGG HOUSE PAINTED WOOD,12 IVORY ROSE PEG PLACE SETTINGS,...,YULETIDE IMAGES GIFT WRAP SET,ZINC HEART T-LIGHT HOLDER,ZINC STAR T-LIGHT HOLDER,ZINC BOX SIGN HOME,ZINC FOLKART SLEIGH BELLS,ZINC HEART LATTICE T-LIGHT HOLDER,ZINC METAL HEART DECORATION,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS SMALL,ZINC WILLIE WINKIE CANDLE STICK
InvoiceNo,,,,,,,,,,,,,,,,,,,,,
536527,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
536840,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
536861,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
536967,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
536983,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
581266,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
581494,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
581570,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Lösche die Spalte "POSTAGE"

In [15]:
subset_df = subset_df.drop("POSTAGE", axis=1)

In einem letzten Schritt müssen die einzelnen Einträge der Matrix in boolesche Werte übertragen werden. Wenn der Wert in einer Zelle kleiner oder gleich 0 ist, wird die Zelle mit einem `False` befüllt, andernfalls mit `True`.

In [16]:
# Definiere eine Funktion, die den Wert False ausgibt, wenn x kleiner-gleich 0, und True, wenn x größer 0
def hot_encode(x):
    if x<= 0:
        return False
    else:
        return True

# Wende die Funktion auf den Datensatz an
subset_df = subset_df.map(hot_encode)

# Zeige Ergebnis
subset_df

Description,50'S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,I LOVE LONDON MINI BACKPACK,RED SPOT GIFT BAG LARGE,SET 2 TEA TOWELS I LOVE LONDON,SPACEBOY BABY GIFT SET,10 COLOUR SPACEBOY PEN,12 COLOURED PARTY BALLOONS,12 EGG HOUSE PAINTED WOOD,12 IVORY ROSE PEG PLACE SETTINGS,...,YULETIDE IMAGES GIFT WRAP SET,ZINC HEART T-LIGHT HOLDER,ZINC STAR T-LIGHT HOLDER,ZINC BOX SIGN HOME,ZINC FOLKART SLEIGH BELLS,ZINC HEART LATTICE T-LIGHT HOLDER,ZINC METAL HEART DECORATION,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS SMALL,ZINC WILLIE WINKIE CANDLE STICK
InvoiceNo,,,,,,,,,,,,,,,,,,,,,
536527,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536840,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536861,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536967,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536983,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
581266,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
581494,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
581570,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


Das Resultat ist eine Matrix mit 513 Reihen und 1786 Spalten:
* Der Index der x Achse (Description) enthält die Rechnungsnummer, die als Transaktionsnummer dient.
* Der Index der y Achse (die Spaltennamen) enthält den Namen des bestellten Produkts.

Dieser Datensatz kann nun mithilfe vorprogrammierter Algorithmen analysiert werden.

## Apriori Algorithmus

### Laden des Pakets

Das Paket `mlxtend` (https://rasbt.github.io/mlxtend/) enthält Funktionen, die den Apriori Algorithmus implementieren.

Wir laden aus dem Modul `frequent_patterns` des Pakets die Funktionen `apriori` und `association_rules`:

In [17]:
from mlxtend.frequent_patterns import apriori, association_rules

### Schritt 1: Finde häufig vorkommende "Itemsets"

Wir setzen den minimalen Support auf 5%. D.h. die Itemsets müssen mindestens in 5% aller Beobachtungen enthalten sein.

In [18]:
frq_items = apriori(subset_df, min_support=0.05, use_colnames=True)

Zeige Ergebnis geordnet nach Support

In [19]:
frq_items.sort_values(['support'], ascending=[False])

,support,itemsets
37,0.237817,(ROUND SNACK BOXES SET OF4 WOODLAND )
36,0.150097,(ROUND SNACK BOXES SET OF 4 FRUITS )
26,0.134503,(PLASTERS IN TIN WOODLAND ANIMALS)
33,0.132554,(REGENCY CAKESTAND 3 TIER)
23,0.124756,(PLASTERS IN TIN CIRCUS PARADE )
56,0.122807,"(ROUND SNACK BOXES SET OF4 WOODLAND , ROUND SN..."
47,0.118908,(WOODLAND CHARLOTTE BAG)
45,0.109162,(SPACEBOY LUNCH BOX )
24,0.103314,(PLASTERS IN TIN SPACEBOY)
10,0.097466,(JAM MAKING SET PRINTED)


### Schritt 2: Finde starke Association Rules

Die Funktion `association_rules` berechnet auf Basis der Ergebnisse im vorherigen Schritt (support > 0.05) die "Confidence". Wir lassen nur jene Werte ausgeben, wo die Confidence mindestens 40% beträgt.

In [20]:
# Berechne die Association Rules
rules = association_rules(frq_items, metric = 'confidence', min_threshold = .4)

# Zeige die Ergebnisse geordnet nach 'confidence'
rules.sort_values(['confidence'], ascending=[False])

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
10,(RED RETROSPOT CHARLOTTE BAG),(WOODLAND CHARLOTTE BAG),0.068226,0.118908,0.058480,0.857143,7.208431,0.050367,6.167641,0.924338
12,(ROUND SNACK BOXES SET OF 4 FRUITS ),(ROUND SNACK BOXES SET OF4 WOODLAND ),0.150097,0.237817,0.122807,0.818182,3.440387,0.087111,4.192008,0.834608
1,(PLASTERS IN TIN STRONGMAN),(PLASTERS IN TIN CIRCUS PARADE ),0.072125,0.124756,0.052632,0.729730,5.849240,0.043634,3.238402,0.893480
13,(SPACEBOY LUNCH BOX ),(ROUND SNACK BOXES SET OF4 WOODLAND ),0.109162,0.237817,0.068226,0.625000,2.628074,0.042266,2.032489,0.695405
2,(PLASTERS IN TIN CIRCUS PARADE ),(PLASTERS IN TIN WOODLAND ANIMALS),0.124756,0.134503,0.072125,0.578125,4.298234,0.055345,2.051549,0.876723
3,(PLASTERS IN TIN WOODLAND ANIMALS),(PLASTERS IN TIN CIRCUS PARADE ),0.134503,0.124756,0.072125,0.536232,4.298234,0.055345,1.887244,0.886596
8,(PLASTERS IN TIN WOODLAND ANIMALS),(ROUND SNACK BOXES SET OF4 WOODLAND ),0.134503,0.237817,0.072125,0.536232,2.254811,0.040138,1.643458,0.642988
6,(PLASTERS IN TIN SPACEBOY),(PLASTERS IN TIN WOODLAND ANIMALS),0.103314,0.134503,0.054581,0.528302,3.927810,0.040685,1.834854,0.831289
11,(ROUND SNACK BOXES SET OF4 WOODLAND ),(ROUND SNACK BOXES SET OF 4 FRUITS ),0.237817,0.150097,0.122807,0.516393,3.440387,0.087111,1.757426,0.930662
14,(WOODLAND CHARLOTTE BAG),(ROUND SNACK BOXES SET OF4 WOODLAND ),0.118908,0.237817,0.060429,0.508197,2.136926,0.032150,1.549773,0.603840


In [21]:
# Zeige die Ergebnisse geordnet nach 'lift'
rules.sort_values(['lift'], ascending=[False])

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
9,(WOODLAND CHARLOTTE BAG),(RED RETROSPOT CHARLOTTE BAG),0.118908,0.068226,0.058480,0.491803,7.208431,0.050367,1.833491,0.977507
10,(RED RETROSPOT CHARLOTTE BAG),(WOODLAND CHARLOTTE BAG),0.068226,0.118908,0.058480,0.857143,7.208431,0.050367,6.167641,0.924338
0,(PLASTERS IN TIN CIRCUS PARADE ),(PLASTERS IN TIN STRONGMAN),0.124756,0.072125,0.052632,0.421875,5.849240,0.043634,1.604973,0.947208
1,(PLASTERS IN TIN STRONGMAN),(PLASTERS IN TIN CIRCUS PARADE ),0.072125,0.124756,0.052632,0.729730,5.849240,0.043634,3.238402,0.893480
3,(PLASTERS IN TIN WOODLAND ANIMALS),(PLASTERS IN TIN CIRCUS PARADE ),0.134503,0.124756,0.072125,0.536232,4.298234,0.055345,1.887244,0.886596
2,(PLASTERS IN TIN CIRCUS PARADE ),(PLASTERS IN TIN WOODLAND ANIMALS),0.124756,0.134503,0.072125,0.578125,4.298234,0.055345,2.051549,0.876723
7,(PLASTERS IN TIN WOODLAND ANIMALS),(PLASTERS IN TIN SPACEBOY),0.134503,0.103314,0.054581,0.405797,3.927810,0.040685,1.509057,0.861245
6,(PLASTERS IN TIN SPACEBOY),(PLASTERS IN TIN WOODLAND ANIMALS),0.103314,0.134503,0.054581,0.528302,3.927810,0.040685,1.834854,0.831289
11,(ROUND SNACK BOXES SET OF4 WOODLAND ),(ROUND SNACK BOXES SET OF 4 FRUITS ),0.237817,0.150097,0.122807,0.516393,3.440387,0.087111,1.757426,0.930662
12,(ROUND SNACK BOXES SET OF 4 FRUITS ),(ROUND SNACK BOXES SET OF4 WOODLAND ),0.150097,0.237817,0.122807,0.818182,3.440387,0.087111,4.192008,0.834608


## FP-Growth

Lade die Funktion `fpgrowth` aus dem Modul `frequent_patterns` des Pakets `mlxtend`:

In [22]:
from mlxtend.frequent_patterns import fpgrowth

Wende den FP-Growth Algorithmus wie oben an.

### Schritt 1: Frequent Itemsets

In [23]:
# Algorithmus
frq_items = fpgrowth(subset_df, min_support=0.05, use_colnames=True)

# Zeige Ergebnis sortiert nach Support an
frq_items.sort_values(['support'], ascending=[False])

,support,itemsets
9,0.237817,(ROUND SNACK BOXES SET OF4 WOODLAND )
10,0.150097,(ROUND SNACK BOXES SET OF 4 FRUITS )
20,0.134503,(PLASTERS IN TIN WOODLAND ANIMALS)
11,0.132554,(REGENCY CAKESTAND 3 TIER)
12,0.124756,(PLASTERS IN TIN CIRCUS PARADE )
49,0.122807,"(ROUND SNACK BOXES SET OF4 WOODLAND , ROUND SN..."
13,0.118908,(WOODLAND CHARLOTTE BAG)
14,0.109162,(SPACEBOY LUNCH BOX )
26,0.103314,(PLASTERS IN TIN SPACEBOY)
1,0.097466,(JAM MAKING SET PRINTED)


### Schritt 2: Finde starke Zusammenhänge

In [24]:
rules = association_rules(frq_items, metric='confidence', min_threshold=.4)

rules.sort_values(['confidence'], ascending=[False])

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
9,(RED RETROSPOT CHARLOTTE BAG),(WOODLAND CHARLOTTE BAG),0.068226,0.118908,0.058480,0.857143,7.208431,0.050367,6.167641,0.924338
1,(ROUND SNACK BOXES SET OF 4 FRUITS ),(ROUND SNACK BOXES SET OF4 WOODLAND ),0.150097,0.237817,0.122807,0.818182,3.440387,0.087111,4.192008,0.834608
14,(PLASTERS IN TIN STRONGMAN),(PLASTERS IN TIN CIRCUS PARADE ),0.072125,0.124756,0.052632,0.729730,5.849240,0.043634,3.238402,0.893480
7,(SPACEBOY LUNCH BOX ),(ROUND SNACK BOXES SET OF4 WOODLAND ),0.109162,0.237817,0.068226,0.625000,2.628074,0.042266,2.032489,0.695405
4,(PLASTERS IN TIN CIRCUS PARADE ),(PLASTERS IN TIN WOODLAND ANIMALS),0.124756,0.134503,0.072125,0.578125,4.298234,0.055345,2.051549,0.876723
5,(PLASTERS IN TIN WOODLAND ANIMALS),(PLASTERS IN TIN CIRCUS PARADE ),0.134503,0.124756,0.072125,0.536232,4.298234,0.055345,1.887244,0.886596
10,(PLASTERS IN TIN WOODLAND ANIMALS),(ROUND SNACK BOXES SET OF4 WOODLAND ),0.134503,0.237817,0.072125,0.536232,2.254811,0.040138,1.643458,0.642988
11,(PLASTERS IN TIN SPACEBOY),(PLASTERS IN TIN WOODLAND ANIMALS),0.103314,0.134503,0.054581,0.528302,3.927810,0.040685,1.834854,0.831289
0,(ROUND SNACK BOXES SET OF4 WOODLAND ),(ROUND SNACK BOXES SET OF 4 FRUITS ),0.237817,0.150097,0.122807,0.516393,3.440387,0.087111,1.757426,0.930662
6,(WOODLAND CHARLOTTE BAG),(ROUND SNACK BOXES SET OF4 WOODLAND ),0.118908,0.237817,0.060429,0.508197,2.136926,0.032150,1.549773,0.603840


In [25]:
# Zeige die Ergebnisse geordnet nach 'lift'
rules.sort_values(['lift'], ascending=[False])

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
8,(WOODLAND CHARLOTTE BAG),(RED RETROSPOT CHARLOTTE BAG),0.118908,0.068226,0.058480,0.491803,7.208431,0.050367,1.833491,0.977507
9,(RED RETROSPOT CHARLOTTE BAG),(WOODLAND CHARLOTTE BAG),0.068226,0.118908,0.058480,0.857143,7.208431,0.050367,6.167641,0.924338
13,(PLASTERS IN TIN CIRCUS PARADE ),(PLASTERS IN TIN STRONGMAN),0.124756,0.072125,0.052632,0.421875,5.849240,0.043634,1.604973,0.947208
14,(PLASTERS IN TIN STRONGMAN),(PLASTERS IN TIN CIRCUS PARADE ),0.072125,0.124756,0.052632,0.729730,5.849240,0.043634,3.238402,0.893480
5,(PLASTERS IN TIN WOODLAND ANIMALS),(PLASTERS IN TIN CIRCUS PARADE ),0.134503,0.124756,0.072125,0.536232,4.298234,0.055345,1.887244,0.886596
4,(PLASTERS IN TIN CIRCUS PARADE ),(PLASTERS IN TIN WOODLAND ANIMALS),0.124756,0.134503,0.072125,0.578125,4.298234,0.055345,2.051549,0.876723
12,(PLASTERS IN TIN WOODLAND ANIMALS),(PLASTERS IN TIN SPACEBOY),0.134503,0.103314,0.054581,0.405797,3.927810,0.040685,1.509057,0.861245
11,(PLASTERS IN TIN SPACEBOY),(PLASTERS IN TIN WOODLAND ANIMALS),0.103314,0.134503,0.054581,0.528302,3.927810,0.040685,1.834854,0.831289
0,(ROUND SNACK BOXES SET OF4 WOODLAND ),(ROUND SNACK BOXES SET OF 4 FRUITS ),0.237817,0.150097,0.122807,0.516393,3.440387,0.087111,1.757426,0.930662
1,(ROUND SNACK BOXES SET OF 4 FRUITS ),(ROUND SNACK BOXES SET OF4 WOODLAND ),0.150097,0.237817,0.122807,0.818182,3.440387,0.087111,4.192008,0.834608


## Mining quantitativer Beziehungen

### Daten

### Algorithmen

## Quellen

* https://developer.ibm.com/tutorials/awb-implementing-apriori-algorithm-python/
* https://rasbt.github.io/mlxtend/user_guide/frequent_patterns/apriori/